<a href="https://colab.research.google.com/github/FlavioRMiranda/EP-02---Assistencia-24h-Guincho/blob/main/EP02_Assistencia_24h_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EP02 — Simulação discreta de assistência 24h / guincho

**SENAC • Computação Científica • Caso 21**  
**Versão de estudo e revisão do grupo — 14/11/2026**

Este notebook estuda como a demanda afeta a espera por mecânicos e guinchos e compara equipes para um serviço hipotético. Não representa dados operacionais de uma seguradora.

**Como usar:** carregue este arquivo no Google Colab e selecione **Ambiente de execução → Executar tudo**. As tabelas, os gráficos e as conclusões são calculados pelo próprio notebook. O estudo de equipes pode levar alguns minutos.

**Responsabilidade acadêmica:** material preparado com apoio de IA. O grupo deve estudar, conferir, adaptar e explicar o modelo e produzir sua interpretação, conforme a política da disciplina. Não apresentar este material como produção independente de assistência. O vídeo será produzido posteriormente pelo grupo.

## 1. Objetivo e perguntas

1. Como filas, espera e ocupação mudam entre um dia normal, maior volume e um pico extremo?
2. Qual recurso limita o atendimento e como o encaminhamento mecânico → guincho afeta sua carga?
3. Qual é a menor equipe, entre as combinações testadas, que cumpre as metas didáticas?
4. Após o pico, os chamados anteriores ao fim do pico são concluídos enquanto novos pedidos continuam chegando?


## 2. Pesquisa: processo real e relação com as filas

**Solicitação e despacho.** A Porto informa que o motorista pode solicitar guincho por aplicativo, serviço on-line ou telefone, fornecendo tipo de serviço, localização e destino [1]. Isso fundamenta a entidade “pedido de socorro”. Neste modelo, cadastro e triagem são instantâneos; a fila começa ao solicitar a equipe.

**Tipos de atendimento.** A página institucional da Porto descreve assistência com técnico ou guincho para ocorrências como problemas mecânicos, elétricos e pneus [2]. O modelo agrega essas ocorrências em dois caminhos, sem simular individualmente cada diagnóstico.

**Reparo seguido de reboque.** As condições do Allianz Auto de janeiro de 2025 descrevem tentativa de reparo emergencial no local em caso de pane mecânica/elétrica e atendimento sujeito às condições contratadas [3]. O caso 21 da disciplina estabelece o encaminhamento ao guincho quando o mecânico não resolve [6]. Por isso um mesmo pedido pode consumir dois serviços em sequência. As probabilidades desse encaminhamento são hipóteses, não estatísticas extraídas da seguradora.

**Variação de movimento.** A PRF relaciona sua operação de Corpus Christi de 2025 ao aumento esperado de veículos nas rodovias [4]. Isso motiva investigar períodos de maior movimento. Converter esse contexto em mais chamados é uma hipótese de modelagem: a fonte não estabelece proporcionalidade entre tráfego e pedidos nem os valores 4, 5 e 8 chamados/hora.

**Problemas de fila investigados.** Com equipes finitas, um pedido espera quando todas as unidades do seu tipo estão ocupadas. Deslocamentos e serviços longos reduzem a disponibilidade; encaminhamentos acrescentam carga aos guinchos. Essas são consequências do modelo de filas adotado, não medições de atraso das empresas citadas. O serviço de guincho da Porto também menciona condições de acesso e liberação policial [7], indicando fatores reais que uma duração agregada não explica separadamente.

As fontes descrevem o funcionamento e o contexto. **Nenhuma delas fornece a calibração numérica usada aqui.** Não houve coleta própria de dados nem observação de uma operação real. Para uso operacional seriam necessários registros anonimizados de chegada, despacho, início/fim, deslocamento, resolução e disponibilidade por região.


## 3. Modelagem: quem é quem

| Conceito | Representação |
|---|---|
| Usuário / entidade / consumidor | Um pedido de socorro de um motorista |
| Servidores | Unidades de atendimento: mecânico com veículo; guincho com operador |
| Filas | Uma para mecânicos e outra para guinchos, ambas FIFO e sem limite de tamanho |
| Recurso | Capacidade simultânea de cada equipe, implementada com `simpy.Resource` |
| Serviço mecânico | Deslocamento + tentativa de reparo, agregado em uma duração |
| Serviço de guincho | Deslocamento + preparação e reboque, agregado em uma duração |
| Eventos | Chegada, entrada na fila, início/fim de serviço, encaminhamento e saída |
| Pausas / turnos | Não modelados; a capacidade é constante e representa unidades simultaneamente disponíveis, não pessoas trabalhando 48h seguidas |
| Crítico | Pedido cuja soma das esperas nas duas filas ultrapassa 60 minutos |
| Desistências | Não modeladas; todos os pedidos são concluídos |

```text
Pedido → classificação instantânea
          ├─ 70% → fila mecânico → deslocamento/reparo
          │                          ├─ 80% resolvido → saída
          │                          └─ 20% não resolvido ─┐
          └─ 30% direto ao guincho ────────────────────────┤
                                                        ↓
                                    fila guincho → deslocamento/reboque → saída
```

**Variação obrigatória:** múltiplas etapas; também há política FIFO explícita. O enunciado exige pelo menos uma opção, não as cinco.

**Escopo e limites.** Uma região agregada; sem prioridade por risco, distâncias explícitas, cancelamentos, oficinas com fila ou falhas. A liberação após o serviço pressupõe disponibilidade imediata para outro pedido; o retorno/reposicionamento não é modelado separadamente. A espera mede tempo até o início da mobilização, não tempo até a chegada física ao motorista. O tempo total inclui esperas, deslocamentos e serviços. O limite de 60 min é uma regra de estudo, não prazo contratual ou legal.


## 4. Parâmetros e origem

Todos os números abaixo são **estimativas didáticas justificadas**, preservando os valores centrais da versão anterior para permitir continuidade do estudo. As referências explicam o processo, não comprovam esses números.

| Parâmetro | Valor | Origem e justificativa |
|---|---|---|
| Chegada normal | 4 pedidos/h | Hipótese para uma operação regional pequena: 96 pedidos esperados por dia |
| Maior volume | 5 pedidos/h | Sensibilidade de +25%, mantendo capacidade média superior à carga |
| Pico extremo | 8 pedidos/h entre 8h e 24h | Teste de estresse: dobra a taxa normal e supera a capacidade inicial |
| Mecânico primeiro | 70% | Hipótese de predominância de ocorrências com tentativa de reparo |
| Sucesso do mecânico | 80% dos encaminhados | Hipótese para testar resolução local; exige calibração real |
| Serviço mecânico | Triangular(20, 40, 75) min | Limites/moda hipotéticos; representa durações positivas com valor mais provável e cauda para serviços demorados |
| Serviço guincho | Triangular(35, 70, 120) min | Hipótese de duração maior por preparação e transporte |
| Equipe inicial | 4 mecânicos e 3 guinchos simultâneos | Configuração de referência a ser avaliada, sem alegar representar uma empresa |
| Espera crítica | Soma das esperas >60 min | Regra didática para medir deterioração, sem desistência |
| Meta de qualidade | Espera média ≤30 min e críticos ≤10% | Critério didático de dimensionamento; não vem do professor ou das seguradoras |
| Janela de chegadas | 48 h | Dois dias para observar pico e recuperação; depois, drenar pendências |
| Repetições | 30 por cenário; 20 na busca; 50 na validação | Compromisso entre custo computacional e avaliação de variação entre execuções |
| Busca de equipes | 2–8 unidades de cada tipo | Domínio finito explicitado; não garante ótimo fora dessa faixa |

### Distribuições e unidades

Em cada trecho de taxa constante, os intervalos são exponenciais com média **60/λ minutos**. Isso corresponde à hipótese de chegadas independentes de Poisson no trecho; não foi feito ajuste a dados. Entre trechos, reiniciamos o sorteio, usando a propriedade sem memória da exponencial.

Para o serviço triangular, `a ≤ moda ≤ b`, média `(a + moda + b)/3`. Assim, o mecânico leva em média 45 min e o guincho 75 min. A triangular é uma escolha exploratória quando se dispõe de limites e moda propostos; não uma distribuição empiricamente validada. Os serviços e os caminhos são independentes dos horários e das filas.

### Configuração dos cenários

| Cenário | 0–8h | 8–24h | 24–48h | Interpretação hipotética |
|---|---:|---:|---:|---|
| Normal | 4/h | 4/h | 4/h | Operação habitual |
| Alta demanda | 5/h | 5/h | 5/h | Movimento maior sustentado |
| Extremo | 4/h | 8/h | 4/h | Pico de viagens, seguido de normalização |

Em todos: mesmas distribuições de serviço, 4 mecânicos/3 guinchos, FIFO, regra crítica de 60 min e nenhuma desistência. O maior volume pode ter capacidade média suficiente e ainda apresentar espera inaceitável; essa distinção será avaliada pelos resultados.

Após 48h não entram novos pedidos; a simulação segue até concluir todos. A recuperação entre 24h e 48h acontece **com novas chegadas**; a drenagem após 48h é outra fase. O sistema começa vazio, sem aquecimento: trata-se de experimento transiente finito, não estimativa de regime permanente.


In [ ]:
%pip -q install simpy==4.1.1 numpy pandas matplotlib


In [ ]:
import copy
from collections import deque
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
HORIZONTE = 48 * 60
LIMITE = 60.0
TEMPO_M = (20, 40, 75)
TEMPO_G = (35, 70, 120)
P_M, P_SUCESSO = .70, .80
N_M, N_G = 4, 3
SEMENTE = 2026
REPS_CENARIOS, REPS_BUSCA, REPS_VALIDACAO = 30, 20, 50
PERFIS = {
    'Normal': [(0, 48, 4)],
    'Alta demanda': [(0, 48, 5)],
    'Extremo': [(0, 8, 4), (8, 24, 8), (24, 48, 4)]
}

def gerar_demanda(perfil, semente):
    # Fluxos separados; sorteios por pedido ocorrem ANTES da simulação.
    streams = np.random.SeedSequence(semente).spawn(5)
    rc, rt, rs, rm, rg = [np.random.default_rng(s) for s in streams]
    chegadas = []
    for inicio_h, fim_h, taxa in perfil:
        t = inicio_h * 60
        while True:
            t += float(rc.exponential(60 / taxa))
            if t >= fim_h * 60:
                break
            chegadas.append(t)
    n = len(chegadas)
    tipos = rt.random(n) < P_M
    sucessos = rs.random(n) < P_SUCESSO
    dm = rm.triangular(*TEMPO_M, size=n)
    dg = rg.triangular(*TEMPO_G, size=n)
    return [dict(id=i+1, chegada_min=t, mecanico=bool(tipos[i]),
                 sucesso=bool(sucessos[i]), dur_m=float(dm[i]), dur_g=float(dg[i]))
            for i, t in enumerate(chegadas)]


## 5. Implementação e medição

O ambiente SimPy avança entre eventos. `request()` aguarda uma unidade e o contexto `with` a libera ao final. Uma lista de identificadores permite medir o trabalho de cada unidade.

O registro contém mudanças das filas, ocupação e população. Eventos no mesmo instante são agregados antes de construir os gráficos em degraus, evitando estados intermediários de duração zero. A marca crítica é registrada no momento em que a espera acumulada cruza o limite, mesmo que o pedido termine depois.

Cada repetição gera previamente a lista completa de pedidos. Todas as equipes comparadas recebem **exatamente essa mesma lista**: mudar capacidade não muda os caminhos ou as durações sorteadas. Repetições distintas usam sementes distintas.


In [ ]:
class SimulacaoAssistencia:
    def __init__(self, nome, demanda, nm=N_M, ng=N_G):
        self.nome, self.nm, self.ng = nome, nm, ng
        self.env = simpy.Environment()
        self.recursos = {'M': simpy.Resource(self.env, capacity=nm),
                         'G': simpy.Resource(self.env, capacity=ng)}
        self.livres = {'M': deque(f'M{i+1}' for i in range(nm)),
                       'G': deque(f'G{i+1}' for i in range(ng))}
        self.demanda = copy.deepcopy(demanda)
        self.pedidos, self.servicos, self.eventos = [], [], []

    def evento(self, **deltas):
        self.eventos.append(dict(tempo_min=self.env.now, **deltas))

    def servir(self, p, tipo):
        recurso = self.recursos[tipo]
        entrada = self.env.now
        self.evento(**{'fila_'+tipo: 1})
        with recurso.request() as req:
            yield req
            espera = self.env.now - entrada
            antes = p['espera_M'] + p['espera_G']
            if p['critico_min'] is None and antes + espera > LIMITE:
                p['critico_min'] = entrada + max(0, LIMITE - antes)
            p['espera_'+tipo] += espera
            servidor = self.livres[tipo].popleft()
            inicio = self.env.now
            self.evento(**{'fila_'+tipo: -1, 'ocup_'+tipo: 1})
            duracao = p['dur_m'] if tipo == 'M' else p['dur_g']
            yield self.env.timeout(duracao)
            self.servicos.append(dict(id=p['id'], tipo=tipo, servidor=servidor,
                                      entrada_min=entrada, inicio_min=inicio,
                                      fim_min=self.env.now, duracao_min=duracao))
            self.evento(**{'ocup_'+tipo: -1})
            self.livres[tipo].append(servidor)

    def atender(self, p):
        if p['mecanico']:
            yield self.env.process(self.servir(p, 'M'))
        if not p['mecanico'] or not p['sucesso']:
            yield self.env.process(self.servir(p, 'G'))
        p['saida_min'] = self.env.now
        p['espera_total_min'] = p['espera_M'] + p['espera_G']
        p['tempo_total_min'] = self.env.now - p['chegada_min']
        self.evento(no_sistema=-1)

    def gerar(self):
        for original in self.demanda:
            yield self.env.timeout(original['chegada_min'] - self.env.now)
            p = dict(original, espera_M=0., espera_G=0., critico_min=None)
            self.pedidos.append(p)
            self.evento(no_sistema=1)
            self.env.process(self.atender(p))
        yield self.env.timeout(HORIZONTE - self.env.now)
        self.evento()

    def executar(self):
        self.evento(no_sistema=0, fila_M=0, fila_G=0, ocup_M=0, ocup_G=0)
        self.env.process(self.gerar())
        self.env.run()
        self.evento()
        self.df = pd.DataFrame(self.pedidos)
        self.sv = pd.DataFrame(self.servicos)
        cols = ['no_sistema', 'fila_M', 'fila_G', 'ocup_M', 'ocup_G']
        eventos = pd.DataFrame(self.eventos).reindex(columns=['tempo_min']+cols).fillna(0)
        self.estados = eventos.groupby('tempo_min')[cols].sum().cumsum().reset_index()
        return self

def indicadores(sim):
    p, e = sim.df, sim.estados
    if p.empty:
        raise ValueError('Nenhum pedido: aumente a janela ou a taxa.')
    r = {'Cenário': sim.nome, 'Pedidos': len(p),
         'Espera média (min)': p.espera_total_min.mean(),
         'P95 espera (min)': p.espera_total_min.quantile(.95),
         'Tempo total médio (min)': p.tempo_total_min.mean(),
         'Críticos (%)': 100 * (p.espera_total_min > LIMITE).mean(),
         'Fim (h)': sim.env.now / 60}
    for h in (24, 48):
        r[f'Pendentes {h}h'] = int(((p.chegada_min <= h*60) & (p.saida_min > h*60)).sum())
    for tipo, quantidade in [('M', sim.nm), ('G', sim.ng)]:
        sv = sim.sv[sim.sv.tipo == tipo] if not sim.sv.empty else sim.sv
        trabalho = sum(max(0., min(s['fim_min'], HORIZONTE)-s['inicio_min'])
                       for s in sv.to_dict('records'))
        r[f'Uso {tipo} (%)'] = 100 * trabalho / (quantidade * HORIZONTE)
    # Limpeza de uma coorte: não equivale a filas permanentemente vazias.
    coorte = p[p.chegada_min < 24*60]
    r['Conclusão coorte pré-24h (h)'] = max(24., coorte.saida_min.max()/60) if len(coorte) else 24.
    for fila in ['M', 'G']:
        r[f'Fila máxima {fila}'] = int(e['fila_'+fila].max())
    return r

def verificar(sim):
    p, e = sim.df, sim.estados
    assert len(p) == len(sim.demanda)
    assert (e.iloc[-1][['no_sistema','fila_M','fila_G','ocup_M','ocup_G']] == 0).all()
    assert (e[['no_sistema','fila_M','fila_G','ocup_M','ocup_G']] >= 0).all().all()
    assert (e.ocup_M <= sim.nm).all() and (e.ocup_G <= sim.ng).all()
    assert (e.no_sistema == e.fila_M+e.fila_G+e.ocup_M+e.ocup_G).all()
    assert np.allclose(p.tempo_total_min,
                       p.espera_total_min + np.where(p.mecanico, p.dur_m, 0)
                       + np.where(~p.mecanico | ~p.sucesso, p.dur_g, 0))
    assert np.array_equal(p.critico_min.notna(), p.espera_total_min > LIMITE)
    for _, grupo in sim.sv.groupby('servidor'):
        g = grupo.sort_values('inicio_min')
        assert (g.inicio_min.to_numpy()[1:] >= g.fim_min.to_numpy()[:-1]-1e-8).all()


## 6. Experimentos: os três cenários

A tabela e os gráficos seguintes mostram uma execução reproduzível de cada cenário, não uma trajetória “média”. Em seguida, 30 repetições mostram a variabilidade. As ocupações usam apenas as primeiras 48h; esperas e tempos totais usam todos os pedidos que chegaram nessa janela, inclusive os concluídos depois.

“Pendentes” inclui pedidos esperando **ou em serviço**. O P95 é o percentil empírico de espera dos pedidos daquela execução. Não há desistências; portanto não é necessário gráfico de desistências.


In [ ]:
resultados = {}
for nome, perfil in PERFIS.items():
    sim = SimulacaoAssistencia(nome, gerar_demanda(perfil, SEMENTE)).executar()
    verificar(sim)
    resultados[nome] = sim
tabela = pd.DataFrame([indicadores(s) for s in resultados.values()])
display(tabela.round(2))
print('Verificações: conservação de pedidos, capacidade, tempos e servidores sem sobreposição — OK.')


In [ ]:
def plotar(sim):
    e, p = sim.estados, sim.df
    x = e.tempo_min / 60
    fig, axs = plt.subplots(3, 2, figsize=(15, 12))
    fig.suptitle(f'Assistência 24h — {sim.nome} | {sim.nm} mecânicos / {sim.ng} guinchos', fontsize=15)
    for tipo, rotulo in [('M', 'Mecânicos'), ('G', 'Guinchos')]:
        axs[0,0].step(x, e['fila_'+tipo], where='post', label=rotulo)
        axs[2,1].step(x, e['ocup_'+tipo], where='post', label=rotulo)
    axs[0,0].set(title='Tamanho das filas', ylabel='Pedidos aguardando')
    axs[0,0].legend()
    axs[0,1].step(x, e.no_sistema, where='post')
    axs[0,1].set(title='Pedidos no sistema', ylabel='Pedidos')
    axs[1,0].scatter(p.id, p.espera_total_min, s=9, label='Espera acumulada')
    axs[1,0].scatter(p.id, p.tempo_total_min, s=9, alpha=.6, label='Tempo total')
    axs[1,0].axhline(LIMITE, color='red', linestyle='--', label='Limite de espera')
    axs[1,0].set(title='Tempo de cada pedido', xlabel='ID do pedido (ordem de chegada)', ylabel='Minutos')
    axs[1,0].legend()
    # Pontos de quebra exatos: a curva sobe apenas durante cada serviço.
    eixo = np.unique(np.r_[0., sim.sv.inicio_min, sim.sv.fim_min, HORIZONTE, sim.env.now])
    for servidor, grupo in sim.sv.groupby('servidor'):
        acum = sum((np.clip(eixo-s.inicio_min, 0, s.fim_min-s.inicio_min)
                    for s in grupo.itertuples()), np.zeros(len(eixo)))
        axs[1,1].plot(eixo/60, acum/60, label=servidor)
    axs[1,1].set(title='Trabalho acumulado por servidor', ylabel='Horas em serviço')
    axs[1,1].legend(ncol=3, fontsize=7)
    criticos = np.sort(p.critico_min.dropna().to_numpy()) / 60
    axs[2,0].step(np.r_[0, criticos, sim.env.now/60],
                  np.r_[0, np.arange(1,len(criticos)+1),len(criticos)], where='post')
    axs[2,0].set(title='Pedidos que ultrapassaram 60 min de espera', ylabel='Críticos acumulados')
    axs[2,1].set(title='Servidores ocupados', ylabel='Unidades em serviço')
    axs[2,1].legend()
    temporais = [axs[0,0], axs[0,1], axs[1,1], axs[2,0], axs[2,1]]
    for ax in temporais:
        ax.set_xlabel('Tempo simulado (h)')
        ax.axvline(48, color='black', linestyle=':', label='Fim das chegadas')
        if sim.nome == 'Extremo':
            ax.axvspan(8, 24, color='orange', alpha=.12)
        ax.set_xlim(0, sim.env.now/60)
    fig.text(.5, .005, 'Faixa laranja: pico de 8 a 24h (extremo). Linha pontilhada: encerramento das chegadas em 48h.', ha='center')
    plt.tight_layout(rect=[0,.025,1,.96])
    plt.show()

for sim in resultados.values():
    plotar(sim)


In [ ]:
linhas = []
for nome, perfil in PERFIS.items():
    for r in range(REPS_CENARIOS):
        sim = SimulacaoAssistencia(nome, gerar_demanda(perfil, 20000+r)).executar()
        verificar(sim)
        linhas.append(dict(indicadores(sim), Repetição=r))
repeticoes = pd.DataFrame(linhas)
metricas = ['Espera média (min)', 'P95 espera (min)', 'Críticos (%)',
            'Uso M (%)', 'Uso G (%)', 'Pendentes 24h', 'Pendentes 48h',
            'Conclusão coorte pré-24h (h)']
resumo = repeticoes.groupby('Cenário', sort=False)[metricas].mean()
display(Markdown('### Médias das 30 repetições'))
display(resumo.round(2))
incerteza = repeticoes.groupby('Cenário', sort=False)['Espera média (min)'].agg(['mean','std','count'])
incerteza['Margem 95% aprox.'] = 1.96 * incerteza['std'] / np.sqrt(incerteza['count'])
display(incerteza.round(2))
fig, axs = plt.subplots(1,2,figsize=(12,4))
nomes = list(PERFIS)
axs[0].bar(nomes, incerteza.loc[nomes,'mean'], yerr=incerteza.loc[nomes,'Margem 95% aprox.'], capsize=5)
axs[0].axhline(30,color='red',linestyle='--')
axs[0].set(title='Espera média e IC aproximado de 95%', ylabel='Minutos')
axs[1].bar(nomes,resumo.loc[nomes,'Críticos (%)'])
axs[1].axhline(10,color='red',linestyle='--')
axs[1].set(title='Fração média de pedidos críticos',ylabel='%')
plt.tight_layout(); plt.show()


### Como interpretar a incerteza e a recuperação

O intervalo mostrado é `média ± 1,96 × desvio entre repetições / √n`, uma aproximação normal para a média das esperas médias de cada execução. Não é intervalo de espera de um motorista, nem garantia de cumprimento das metas. Os percentuais são médias dos percentuais por repetição, sem ponderar pelo número de pedidos.

A “conclusão da coorte pré-24h” é o instante em que termina o último pedido chegado antes de 24h, limitado inferiormente a 24h. No extremo, mede a limpeza dos pedidos do período anterior ao fim do pico. Se ocorrer até 48h, essa coorte foi concluída enquanto novas chegadas continuavam. Isso não significa que todas as filas permaneçam vazias, nem que novos chamados não tenham esperado. Os gráficos mostram a evolução completa.


## 7. Checagem analítica da capacidade

Em média, 70% usam mecânico e `30% + 70% × 20% = 44%` usam guincho. Para um trecho de taxa constante, a carga oferecida é `ρM = λ × 0,70 × 45 / (60 × nM)` e `ρG = λ × 0,44 × 75 / (60 × nG)`.

Esse cálculo não substitui a simulação. Carga abaixo de 100% é condição de capacidade média, não promessa de fila pequena; acima de 100%, a demanda sustentada supera a capacidade. O fluxo encaminhado ao guincho pode chegar com atraso e não é necessariamente Poisson. Não aplicamos fórmulas M/M/c ao serviço triangular.


In [ ]:
carga = pd.DataFrame([{'Taxa (pedidos/h)': taxa,
                       'Carga M (%)': 100*taxa*P_M*np.mean(TEMPO_M)/(60*N_M),
                       'Carga G (%)': 100*taxa*(1-P_M*P_SUCESSO)*np.mean(TEMPO_G)/(60*N_G)}
                      for taxa in [4,5,8]])
display(carga.round(2))


## 8. Dimensionamento: comparação controlada de equipes

São testadas 49 combinações (2 a 8 de cada tipo) em cada cenário. Para cada repetição, todas recebem a mesma demanda. A seleção considera **as médias de 20 repetições**: espera média ≤30 min e críticos ≤10%. Minimizamos o número total de unidades; em empate, a espera média. Não minimizamos custo monetário: faltam custos de mecânicos e guinchos.

Depois, a equipe selecionada passa por 50 repetições com sementes novas. O resultado é a menor equipe encontrada **nesse domínio e com esses critérios**, não uma configuração universalmente ideal. A validação é uma avaliação adicional, sem garantia operacional; eventual falha será explicitamente apresentada e impedirá uma recomendação definitiva.


In [ ]:
busca = []
for nome, perfil in PERFIS.items():
    demandas = [gerar_demanda(perfil, 30000+r) for r in range(REPS_BUSCA)]
    for nm in range(2,9):
        for ng in range(2,9):
            valores = [indicadores(SimulacaoAssistencia(nome,d,nm,ng).executar()) for d in demandas]
            busca.append({'Cenário':nome,'Mecânicos':nm,'Guinchos':ng,'Total':nm+ng,
                          'Espera média (min)':np.mean([v['Espera média (min)'] for v in valores]),
                          'Críticos (%)':np.mean([v['Críticos (%)'] for v in valores])})
    print(f'Busca concluída: {nome}')
comparacao = pd.DataFrame(busca)
comparacao['Atende metas'] = (comparacao['Espera média (min)']<=30) & (comparacao['Críticos (%)']<=10)
escolhas = []
for nome in PERFIS:
    elegiveis = comparacao[(comparacao['Cenário']==nome) & comparacao['Atende metas']]
    if not elegiveis.empty:
        escolhas.append(elegiveis.sort_values(['Total','Espera média (min)','Mecânicos']).iloc[0].to_dict())
    else:
        print(f'{nome}: nenhuma equipe elegível. Amplie a busca antes de recomendar.')
selecionadas = pd.DataFrame(escolhas)
display(selecionadas.round(2))
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, nome in zip(axs, PERFIS):
    sub = comparacao[comparacao['Cenário']==nome]
    matriz = sub.pivot(index='Mecânicos',columns='Guinchos',values='Espera média (min)')
    im = ax.imshow(matriz.to_numpy(), origin='lower', aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(7),range(2,9)); ax.set_yticks(range(7),range(2,9))
    ax.set(title=nome,xlabel='Guinchos',ylabel='Mecânicos')
    for v in sub[sub['Atende metas']].itertuples():
        ax.text(v.Guinchos-2,v.Mecânicos-2,'✓',ha='center',va='center',fontsize=10)
    fig.colorbar(im,ax=ax,label='Espera média (min)')
plt.tight_layout(); plt.show()

validacao = []
for escolha in escolhas:
    nome, nm, ng = escolha['Cenário'],int(escolha['Mecânicos']),int(escolha['Guinchos'])
    vs = []
    for r in range(REPS_VALIDACAO):
        sim = SimulacaoAssistencia(nome, gerar_demanda(PERFIS[nome],40000+r), nm, ng).executar()
        verificar(sim)
        vs.append(indicadores(sim))
    v = pd.DataFrame(vs)
    espera, crit = v['Espera média (min)'].mean(),v['Críticos (%)'].mean()
    validacao.append({'Cenário':nome,'Mecânicos':nm,'Guinchos':ng,
                      'Espera média (min)':espera,'Críticos (%)':crit,
                      'Margem 95% espera':1.96*v['Espera média (min)'].std()/np.sqrt(len(v)),
                      'Margem 95% críticos':1.96*v['Críticos (%)'].std()/np.sqrt(len(v)),
                      'Cumpre metas na validação':bool(espera<=30 and crit<=10),
                      'Repetições que cumprem (%)':100*((v['Espera média (min)']<=30)&(v['Críticos (%)']<=10)).mean()})
validacao = pd.DataFrame(validacao)
display(validacao.round(2))


## 9. Conclusões calculadas e discussão

As conclusões numéricas seguintes são geradas com os resultados desta execução. Não reutilizam os números da versão anterior: o horizonte e o perfil extremo foram modificados. Critérios de aprovação são aplicados aos números completos, antes de arredondar.


In [ ]:
texto = ['### Resultados dos cenários com a equipe inicial']
for nome in PERFIS:
    r = resumo.loc[nome]
    atende = r['Espera média (min)']<=30 and r['Críticos (%)']<=10
    texto.append(f"- **{nome}:** espera média {r['Espera média (min)']:.2f} min; "
                 f"críticos {r['Críticos (%)']:.2f}%; uso M {r['Uso M (%)']:.1f}% e G {r['Uso G (%)']:.1f}%. "
                 f"A equipe inicial {'atende' if atende else 'não atende'} às metas nas médias das repetições.")
r = resumo.loc['Extremo']
texto.append(f"\nNo extremo, a conclusão média da coorte anterior a 24h ocorre em {r['Conclusão coorte pré-24h (h)']:.2f}h. "
             "A média não garante que todas as execuções recuperem nesse instante; consulte a trajetória e a dispersão.")
v = repeticoes[repeticoes['Cenário']=='Extremo']
texto.append(f"Em {100*(v['Conclusão coorte pré-24h (h)']<=48).mean():.1f}% das repetições, "
             'essa coorte foi inteiramente concluída até 48h, com novas chegadas ainda ativas.')
texto.append('\n### Configuração recomendável dentro do estudo')
for r in validacao.to_dict('records'):
    status = ('cumpriu as metas médias na validação independente' if r['Cumpre metas na validação'] else
              'não confirmou as metas médias na validação; não é recomendação final e exige nova busca')
    texto.append(f"- **{r['Cenário']}: {r['Mecânicos']} mecânicos e {r['Guinchos']} guinchos** — "
                 f"{status}. Espera {r['Espera média (min)']:.2f} min; críticos {r['Críticos (%)']:.2f}%. "
                 f"{r['Repetições que cumprem (%)']:.0f}% das repetições cumpriram ambas as metas.")
texto.append('\nO guincho recebe os pedidos diretos e os encaminhados e tem serviço médio maior. '
             'O cálculo de carga identifica sua pressão potencial; aumentar só mecânicos não elimina a falta de capacidade de reboque. '
             'No cenário extremo, a taxa de pico supera a capacidade média inicial de ambos os recursos. '
             'Filas separadas por tipo e atendimento em etapas são coerentes com o fluxo adotado. '
             'As equipes selecionadas são alternativas sob hipóteses didáticas, não dimensionamento comprovado de uma empresa.')
display(Markdown('\n\n'.join(texto)))


### Limitações e próximos estudos

- As fontes fundamentam o processo, mas a calibração depende de dados reais ainda não coletados. Alterar taxas, serviços e sucesso pode mudar a equipe escolhida.
- Não há prioridades por risco, fila máxima, desistências, férias, pausas, falhas ou distribuição espacial. A FIFO é uma simplificação que não deve ser tomada como regra de segurança para ocorrências reais.
- Começar vazio pode reduzir a espera inicial. Encerrar chegadas em 48h facilita a drenagem final. Nenhum desses resultados comprova regime permanente.
- Selecionar pelas médias pode esconder dias ruins. P95, percentuais críticos, incerteza e frequência de cumprimento devem ser considerados conjuntamente. Uma regra mais conservadora pode exigir limites superiores dos intervalos abaixo das metas.
- Unidades de mecânico e guincho têm custos diferentes não estimados. A soma de unidades é somente um objetivo didático.
- A busca usa a mistura 70%/80% fixa. Recomenda-se testar sucesso mecânico e tempos de deslocamento, além de validar em registros de outra janela temporal.

## 10. Referências

Referências consultadas em **14 nov. 2026**. Páginas comerciais são evidência do serviço descrito, não auditoria independente de qualidade. Textos abaixo são sínteses, sem reprodução extensa das fontes.

[1] PORTO SEGURO. **Como solicitar um guincho?** [s.d.]. Disponível em: https://www.portoseguro.com.br/faqs/como-solicitar-um-guincho. Acesso em: 14 nov. 2026.

[2] PORTO SEGURO. **Seguro Auto — Cotar Seguro de Carro Online.** [s.d.]. Disponível em: https://www.portoseguro.com.br/sites/institucional/seguro-auto. Acesso em: 14 nov. 2026.

[3] ALLIANZ SEGUROS. **Condições Gerais do Seguro Allianz Auto.** Versão jan. 2025. Seção de assistência 24 horas/reparo emergencial. Disponível em: https://www.allianz.com.br/content/dam/onemarketing/iberolatam/allianz-br/doc-para-links/documento-auto-/Condicoes-Gerais-do-Seguro-Allianz%20Auto_01.2025.pdf. Acesso em: 14 nov. 2026. Documento localizado por pesquisa indexada; a ferramenta de consulta não disponibilizou a leitura integral do PDF, portanto não se atribuem a ele parâmetros numéricos.

[4] BRASIL. POLÍCIA RODOVIÁRIA FEDERAL. **PRF celebra 17 anos da Lei Seca em operação para o feriado de Corpus Christi e festejos juninos.** Jun. 2025. Disponível em: https://www.gov.br/prf/pt-br/noticias/nacionais/2025/junho/prf-celebra-17-anos-da-lei-seca-em-operacao-para-o-feriado-de-corpus-christi-e-festejos-juninos. Acesso em: 14 nov. 2026. Informação sobre aumento esperado do fluxo consultada no resultado indexado da página.

[5] CRIVELARO, Celso. **EP 2 — Simulação Discreta.** Repositório senac-computacao-cientifica, [s.d.]. Disponível em: https://github.com/celsocrivelaro/senac-computacao-cientifica/blob/main/eps/ep02/enunciado.md. Acesso em: 14 nov. 2026.

[6] CRIVELARO, Celso. **EP02 — Caso: Assistência 24h / guincho.** Repositório senac-computacao-cientifica, [s.d.]. Disponível em: https://github.com/celsocrivelaro/senac-computacao-cientifica/blob/main/eps/ep02/casos/21-assistencia-24h-guincho.md. Acesso em: 14 nov. 2026.

[7] PORTO SEGURO. **Guincho.** [s.d.]. Disponível em: https://www.portoseguro.com.br/servicos/guincho. Acesso em: 14 nov. 2026.

[8] SIMPY AUTHORS. **Shared Resources.** Documentação SimPy, [s.d.]. Disponível em: https://simpy.readthedocs.io/en/latest/topical_guides/resources.html. Acesso em: 14 nov. 2026.

[9] CRIVELARO, Celso. **Política de uso de ferramentas generativas de IA.** [s.d.]. Disponível em: https://github.com/celsocrivelaro/senac-computacao-cientifica/blob/main/politicas/ia-generativa.md. Acesso em: 14 nov. 2026.


## 11. Preparação do vídeo e entrega

O vídeo será gravado pelo grupo, publicado no YouTube como **Não listado**, com **5 a 15 minutos**, sem marcar “conteúdo para crianças”. Inserir o link no local de entrega indicado pelo professor. O notebook é o arquivo de código a entregar.

Roteiro sugerido de 8–10 minutos:

1. **0–1 min:** problema, perguntas e pesquisa; distinguir contexto real de números hipotéticos.
2. **1–3 min:** diagrama, filas FIFO, recursos e passagem mecânico → guincho; explicar distribuições.
3. **3–6 min:** demonstrar as células e interpretar gráficos dos três cenários; distinguir espera, serviço e tempo total.
4. **6–8 min:** gargalo, repetição dos experimentos, equipe selecionada e validação.
5. **8–10 min:** recuperação após o pico, limitações e conclusão do grupo.

Antes de entregar: revisar autoria e entendimento; adaptar o texto ao trabalho do grupo; executar tudo no Colab; confirmar as conclusões exibidas; salvar o `.ipynb` com suas saídas. Se parâmetros forem alterados, executar tudo novamente.
